In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# import from biogeme for choice modelling
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme.expressions import Beta, Variable, log, exp
from biogeme import models

In [3]:
# Get the current working directory and adapt the paths if needed
current_dir = Path.cwd()

## 1  Data loading

* `papers_cleaned.csv` paper‑level availability features.  


In [ ]:
DATA_ROOT = current_dir.parents[2] / "zenodo" / "analysis_dataset"
PAPER_CSV  = DATA_ROOT / "papers_cleaned_with_new_columns.csv"

papers   = pd.read_csv(PAPER_CSV, low_memory=False)

In [5]:
# print the dataset to check that it is correctly loaded
papers # 10724 papers and 77 variables

,year,journal,article_subtype,doi,num_authors,primary_institution,number_of_keywords,open_access,figure_number,table_number,...,journal_rate_all,journal_n,papers_is_data_used_true,papers_is_data_used_share,papers_is_data_cited_or_linked_share,papers_atleast_one_dataset_available_share,papers_all_datasets_available_share,papers_uses_data_share,papers_has_code_any_share,is_quantitative_study
0,2019,Transportation Research Part A: Policy and Pra...,fla,10.1016/j.tra.2017.10.006,2,"California PATH Program, University of Califor...",3,False,0,0,...,0.02524,1664,1564,0.939904,0.335337,0.025841,0.02524,0.939904,1.0,True
1,2019,Transportation Research Part A: Policy and Pra...,fla,10.1016/j.tra.2017.10.012,4,Department of Civil and Environmental Engineer...,3,False,5,4,...,0.02524,1664,1564,0.939904,0.335337,0.025841,0.02524,0.939904,1.0,True
2,2019,Transportation Research Part A: Policy and Pra...,fla,10.1016/j.tra.2017.11.003,4,"Department of Planning and Transport, Faculty ...",0,True,0,3,...,0.02524,1664,1564,0.939904,0.335337,0.025841,0.02524,0.939904,1.0,True
3,2019,Transportation Research Part A: Policy and Pra...,fla,10.1016/j.tra.2018.01.023,4,Centre for Mobility and Spatial Planning – Dep...,4,False,3,4,...,0.02524,1664,1564,0.939904,0.335337,0.025841,0.02524,0.939904,1.0,True
4,2019,Transportation Research Part A: Policy and Pra...,fla,10.1016/j.tra.2018.01.025,2,"College of Public Administration, Huazhong Agr...",0,False,12,3,...,0.02524,1664,1564,0.939904,0.335337,0.025841,0.02524,0.939904,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10719,2024,Transportation Research Interdisciplinary Pers...,fla,10.1016/j.trip.2024.101244,2,"Department of Civil Engineering, University of...",5,True,13,3,...,0.03125,1056,968,0.916667,0.336174,0.031250,0.03125,0.916667,1.0,True
10720,2024,Transportation Research Interdisciplinary Pers...,fla,10.1016/j.trip.2024.101245,2,"Lyndon B. Johnson School of Public Affairs, Th...",5,True,4,2,...,0.03125,1056,968,0.916667,0.336174,0.031250,0.03125,0.916667,1.0,True
10721,2024,Transportation Research Interdisciplinary Pers...,fla,10.1016/j.trip.2024.101246,1,"Department of Biology, The University of Dodoma",3,True,2,0,...,0.03125,1056,968,0.916667,0.336174,0.031250,0.03125,0.916667,1.0,True
10722,2024,Transportation Research Interdisciplinary Pers...,fla,10.1016/j.trip.2024.101247,3,"Graduate School of Engineering, Kochi Universi...",6,True,19,7,...,0.03125,1056,968,0.916667,0.336174,0.031250,0.03125,0.916667,1.0,True


# 2 Data processing and descriptive statistics

In [6]:
# Basic information about the dataset
print("\nDataset information:")
print(papers.info())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10724 entries, 0 to 10723
Data columns (total 77 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   year                                        10724 non-null  int64  
 1   journal                                     10724 non-null  object 
 2   article_subtype                             10724 non-null  object 
 3   doi                                         10724 non-null  object 
 4   num_authors                                 10724 non-null  int64  
 5   primary_institution                         10571 non-null  object 
 6   number_of_keywords                          10724 non-null  int64  
 7   open_access                                 10724 non-null  bool   
 8   figure_number                               10724 non-null  int64  
 9   table_number                                10724 non-null  i

In [7]:
# Filter based on is_data_repository_available not null

papers = papers[
    (papers['is_data_repository_available'].notnull())
]

# Create a copy of the dataset before encoding
papers_copy = papers.copy()

# Get dummies for the years, the journal, the topics, and the regions
cols_to_encode = ["year", "journal", 'lda_topic', 'region_normalized']

# One-hot encode only selected variables
papers = pd.get_dummies(papers_copy, columns=cols_to_encode, drop_first=False)

In [8]:
# Recode all the boolean variables to binary variables (TRUE = 1)
bool_cols = papers.select_dtypes(include='bool').columns
papers[bool_cols] = papers[bool_cols].astype(int)

In [9]:
# Code a variable which is a combination of "is_data_repository_available" and "is_data_cited_or_linked"
# Convert object/string columns to binary 0/1

# Convert to string, strip spaces, lower case, then map to 0/1
papers["is_data_repository_available_bin"] = papers["is_data_repository_available"].astype(str).str.strip().str.lower().map({"true":1, "false":0})
papers["is_data_cited_or_linked_bin"] = papers["is_data_cited_or_linked"].astype(str).str.strip().str.lower().map({"true":1, "false":0})

# Check if any values failed to map
print(papers["is_data_repository_available_bin"])
print(papers["is_data_cited_or_linked_bin"])

# Create combined variable 1-4: True/True is 4 and False/False is 1
papers["data_available"] = 1 + 2 * papers["is_data_repository_available_bin"] +  papers["is_data_cited_or_linked_bin"]

0        0
1        0
2        0
3        0
4        0
        ..
10719    0
10720    0
10721    0
10722    0
10723    0
Name: is_data_repository_available_bin, Length: 10723, dtype: int64
0        1
1        1
2        0
3        0
4        1
        ..
10719    1
10720    0
10721    0
10722    0
10723    0
Name: is_data_cited_or_linked_bin, Length: 10723, dtype: int64


In [10]:
# select only the columns relevant for analysis

num_columns = ["num_authors", "citation_count", "paper_age_years",
               "figure_number", "table_number", "reference_count", "page_count", "review_time_days"          
               ] # they can be also considered categorical

cat_columns_man = [ "is_quantitative_study", "is_code_publicly_available", "data_available",
                   "is_data_repository_available_bin", "is_data_cited_or_linked_bin", "uses_data", "open_access",
               ]

# automatically detect by prefix
cat_columns_pref = [col for col in papers.columns if col.startswith(("year_", "lda_", "journal_T", "region_"))]

# combine both
cat_columns = cat_columns_man + cat_columns_pref 

cat_num_columns = cat_columns + num_columns

# keep only these columns
data_orig_v2 = papers
data = papers[cat_num_columns]

# 1. Choice model predicting the availability of code

In [11]:
# we develop a binary logit model to predict the code availability based on the variables available
data = data.copy()

# create binary variables when values are missing
for col in data.columns:
    if data[col].isna().any():
        data[f'{col}_missing'] = data[col].isna().astype(int)

# then fill in all missing values in the dataset
data_bio = data.fillna(0)

data_bio = data_bio.copy()

# read the data as Biogeme dataset
database = db.Database('data_v2', data_bio)

In [12]:
# When we filter on quantitative study, the percentage of code publicly available is 5.0%
data_bio_code_av = data_bio[(data_bio['is_quantitative_study'] == 1)]

data_bio_code_av.describe()

,is_quantitative_study,is_code_publicly_available,data_available,is_data_repository_available_bin,is_data_cited_or_linked_bin,uses_data,open_access,year_2019,year_2020,year_2021,...,region_normalized_South America,num_authors,citation_count,paper_age_years,figure_number,table_number,reference_count,page_count,review_time_days,review_time_days_missing
count,10480.0,10480.000000,10480.000000,10480.000000,10480.000000,10480.000000,10480.000000,10480.000000,10480.000000,10480.000000,...,10480.000000,10480.000000,10480.000000,10480.000000,10480.00000,10480.000000,10480.000000,10480.000000,10480.000000,10480.000000
mean,1.0,0.050382,1.358588,0.035496,0.287595,0.895229,0.268130,0.149809,0.164885,0.175573,...,0.019752,3.755248,28.345324,3.425954,8.74666,6.393607,58.195706,18.442748,223.561260,0.123569
std,0.0,0.218742,0.585020,0.185039,0.452663,0.306273,0.443007,0.356901,0.371095,0.380474,...,0.139153,1.718561,43.196492,1.702306,6.12098,4.499848,26.592660,6.287568,162.465454,0.329105
min,1.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000,0.000000,1.000000,0.00000,0.000000,0.000000,2.000000,0.000000,0.000000
25%,1.0,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,3.000000,7.000000,2.000000,4.00000,3.000000,41.000000,14.000000,125.000000,0.000000
50%,1.0,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,4.000000,16.000000,3.000000,8.00000,6.000000,54.000000,18.000000,205.000000,0.000000
75%,1.0,0.000000,2.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,5.000000,35.000000,5.000000,12.00000,8.000000,69.000000,22.000000,298.000000,0.000000
max,1.0,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,35.000000,1621.000000,6.000000,52.00000,58.000000,365.000000,62.000000,2246.000000,1.000000


In [13]:
# we prepare the variables we want to analyse
# numerical variables are centred in the mean value

at_least_one_dataset_available = Variable('at_least_one_dataset_available')
at_least_one_dataset_available_missing = Variable('at_least_one_dataset_available_missing')

is_quantitative_study = Variable('is_quantitative_study')
is_quantitative_study_missing = Variable('is_quantitative_study_missing')

is_code_publicly_available = Variable('is_code_publicly_available')
is_code_publically_available_missing =  Variable('is_code_publically_available_missing')

uses_data = Variable('uses_data')
uses_data_missing =  Variable('uses_data_missing')

number_of_authors_cent = Variable('num_authors') - data_bio_code_av['num_authors'].mean()
paper_age_years_cent = Variable('paper_age_years') - data_bio_code_av['paper_age_years'].mean()
open_access = Variable('open_access')

figure_number_cent = Variable('figure_number') - data_bio_code_av['figure_number'].mean() 
table_number_cent = Variable('table_number') - data_bio_code_av['table_number'].mean()
reference_count_cent = Variable('reference_count') - data_bio_code_av['reference_count'].mean()
page_count_cent = Variable('page_count') - data_bio_code_av['page_count'].mean()

review_time_days_cent = Variable('review_time_days') - data_bio_code_av[data_bio_code_av['review_time_days'] != 0]['review_time_days'].mean()
review_time_days_missing = Variable('review_time_days_missing')

journal_tra = Variable('journal_Transportation Research Part A: Policy and Practice')
journal_trb = Variable('journal_Transportation Research Part B: Methodological')
journal_trc = Variable('journal_Transportation Research Part C: Emerging Technologies')
journal_trd = Variable('journal_Transportation Research Part D: Transport and Environment')
journal_tre = Variable('journal_Transportation Research Part E: Logistics and Transportation Review')
journal_trf = Variable('journal_Transportation Research Part F: Traffic Psychology and Behaviour')
journal_trip = Variable('journal_Transportation Research Interdisciplinary Perspectives')

Africa = Variable('region_normalized_Africa')
Asia = Variable('region_normalized_Asia')
Europe = Variable('region_normalized_Europe')
NorthAmerica = Variable('region_normalized_North America')
Oceania = Variable('region_normalized_Oceania')
SouthAmerica = Variable('region_normalized_South America')

AirFreight = Variable('lda_topic_Air & Freight Logistics')
AutoDriving = Variable('lda_topic_Automated Driving & Human Factors')
Covid19 = Variable('lda_topic_COVID-19 Impact on Travel & Activities')    
DataDriven = Variable('lda_topic_Data-Driven Modeling & Prediction')
DrivBeh = Variable('lda_topic_Driver Behavior & Safety Risk')
ElectVeh = Variable('lda_topic_Electric Vehicles & Ride-Sharing')
Optimization = Variable('lda_topic_Optimization & Routing Algorithms')   # baseline? 
PubTrans = Variable('lda_topic_Public Transit Service & Demand')
RoadInfra = Variable('lda_topic_Road Infrastructure & Emergency Management')
SocPoli = Variable('lda_topic_Social & Policy Aspects of Mobility')
SuppChain = Variable('lda_topic_Supply Chain & Market Strategies')
TrafficFlow = Variable('lda_topic_Traffic Flow & Network Control')
TransEmiss = Variable('lda_topic_Transportation Emissions & Policy')
TravMode = Variable('lda_topic_Travel & Mode Choice Behavior')
UrbEnv = Variable('lda_topic_Urban Environment & Active Transport')
Other = AirFreight + RoadInfra + SuppChain

In [14]:
# we exclude the studies that are not quantitative
exclude = (is_quantitative_study != 1)
database.remove(exclude)

In [15]:
# we can see the current dataset: 10480 observations, 51 columns
database.data

,is_quantitative_study,is_code_publicly_available,data_available,is_data_repository_available_bin,is_data_cited_or_linked_bin,uses_data,open_access,year_2019,year_2020,year_2021,...,region_normalized_South America,num_authors,citation_count,paper_age_years,figure_number,table_number,reference_count,page_count,review_time_days,review_time_days_missing
0,1,0,2,0,1,0,0,1,0,0,...,0,2,41,6.0,0,0,16,9,0.0,1
1,1,0,2,0,1,1,0,1,0,0,...,0,4,71,6.0,5,4,37,12,0.0,1
2,1,0,1,0,0,1,1,1,0,0,...,0,4,54,6.0,0,3,50,11,193.0,0
3,1,0,1,0,0,1,0,1,0,0,...,0,4,23,6.0,3,4,53,12,0.0,1
4,1,0,2,0,1,1,0,1,0,0,...,0,2,2,6.0,12,3,45,16,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10719,1,0,2,0,1,1,1,0,0,0,...,0,2,1,1.0,13,3,71,15,272.0,0
10720,1,0,1,0,0,1,1,0,0,0,...,0,2,0,1.0,4,2,37,10,206.0,0
10721,1,0,1,0,0,1,1,0,0,0,...,0,1,0,1.0,2,0,25,9,167.0,0
10722,1,0,1,0,0,1,1,0,0,0,...,0,3,0,1.0,19,7,33,17,97.0,0


In [16]:
# MODEL PARAMETERS

asc_ca = Beta('asc_ca', 0, None, None, 0)

beta_year = Beta('beta_year', 0, None, None, 0)

beta_number_of_authors = Beta('beta_number_of_authors', 0, None, None, 0)
beta_paper_age_years = Beta('beta_paper_age_years', 0, None, None, 0)
beta_open_access = Beta('beta_open_access', 0, None, None, 0)

beta_figure_number = Beta('beta_figure_number',  0, None, None, 0 )
beta_table_number = Beta('beta_table_number',  0, None, None, 0)
beta_reference_count = Beta('beta_reference_count',  0, None, None, 0)
beta_page_count = Beta('beta_page_count',  0, None, None, 0)

beta_TRA = Beta('beta_TRA', 0, None, None, 0) 
beta_TRB = Beta('beta_TRB', 0, None, None, 0)
beta_TRB_TRC = Beta('beta_TRB_TRC', 0, None, None, 0)
beta_TRB_TRC_TRE = Beta('beta_TRB_TRC_TRE', 0, None, None, 0)
beta_TRC = Beta('beta_TRC', 0, None, None, 0)
# beta_TRD = Beta('beta_TRD', 0, None, None, 0) reference category has the most common 
beta_TRF_TRIP = Beta('beta_TRF_TRIP', 0, None, None, 0)
beta_TRB_TRC_TRE_TRIP = Beta('beta_TRB_TRC_TRE_TRIP', 0, None, None, 0)
beta_TRE = Beta('beta_TRE', 0, None, None, 0)
beta_TRF = Beta('beta_TRF', 0, None, None, 0)
beta_TRIP = Beta('beta_TRIP', 0, None, None, 0)

beta_Africa = Beta('beta_Africa', 0, None, None, 0)
#beta_Asia = Beta('beta_Asia', 0, None, None, 0) reference category
beta_noAsia = Beta('beta_noAsia', 0, None, None, 0)
beta_Europe = Beta('beta_Europe', 0, None, None, 0)
beta_NorthAmerica = Beta('beta_NorthAmerica', 0, None, None, 0)
beta_NorthAmerica_Oceania = Beta('beta_NorthAmerica_Oceania', 0, None, None, 0)
beta_Europe_SouthAmerica_Africa = Beta('beta_Europe_SouthAmerica_Africa', 0, None, None, 0)
beta_Europe_NorthAmerica = Beta('beta_Europe_NorthAmerica', 0, None, None, 0)
beta_Oceania= Beta('beta_Oceania', 0, None, None, 0)
beta_SouthAmerica = Beta('beta_SouthAmerica', 0, None, None, 0)
beta_SouthAmerica_Africa = Beta('beta_SouthAmerica_Africa', 0, None, None, 0)

beta_other_topics = Beta('beta_other_topics', 0, None, None, 0)
beta_AutoDriving = Beta('beta_AutoDriving', 0, None, None, 0)
beta_Covid19 = Beta('beta_Covid19', 0, None, None, 0)  
beta_DataDriven = Beta('beta_DataDriven', 0, None, None, 0)
beta_DrivBeh = Beta('beta_DrivBeh', 0, None, None, 0)
beta_ElectVeh = Beta('beta_ElectVeh', 0, None, None, 0)
#beta_Optimization = Beta('beta_Optimization', 0, None, None, 0) reference category
beta_PubTrans = Beta('beta_PubTrans', 0, None, None, 0)
beta_SocPoli = Beta('beta_SocPoli', 0, None, None, 0)
beta_TrafficFlow = Beta('beta_TrafficFlow', 0, None, None, 0)
beta_TransEmiss = Beta('beta_TransEmiss', 0, None, None, 0)
beta_TravMode = Beta('beta_TravMode', 0, None, None, 0)
beta_UrbEnv = Beta('beta_UrbEnv', 0, None, None, 0)
beta_other_topics_DrivBeh = Beta('beta_other_topics_DrivBeh', 0, None, None, 0 )
beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode = Beta('beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode', 0, None, None, 0)

## 1.1 Baseline model with only a constant

In [17]:
# Model specification
v_code_av = (asc_ca
    )

v_no_code_av = 0

V = {1: v_code_av, 0: v_no_code_av }
av = {1: 1, 0: 1}

logprob = models.loglogit(V, av, is_code_publicly_available)
biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'logit_const_code'

File biogeme.toml has been created


In [18]:
# Estimate the model
results_const_code_av = biogeme.estimate()

print(results_const_code_av.printGeneralStatistics())


Number of estimated parameters:	1
Sample size:	10480
Excluded observations:	243
Init log likelihood:	-7264.182
Final log likelihood:	-2092.201
Likelihood ratio test for the init. model:	10343.96
Rho-square for the init. model:	0.712
Rho-square-bar for the init. model:	0.712
Akaike Information Criterion:	4186.403
Bayesian Information Criterion:	4193.66
Final gradient norm:	9.2135E-06
Nbr of threads:	10



In [19]:
results_const_code_av.getEstimatedParameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_ca,-2.936433,0.044659,-65.752389,0.0


## 1.2 Final model 

In [20]:
# MODEL SPECIFICATION

v_code_av = (asc_ca # + beta_year * year_cent
    #+ beta_number_of_authors * number_of_authors
    + beta_paper_age_years * paper_age_years_cent 
    + beta_figure_number * figure_number_cent
    #+ beta_table_number * table_number
    #+ beta_reference_count * reference_count
   # + beta_page_count * page_count
    #+ beta_TRA * journal_tra
    + beta_TRB_TRC * (journal_trb + journal_trc)
    #+ beta_TRC * journal_trc 
    #+ beta_TRE * journal_tre 
    #+ beta_TRF_TRIP * (journal_trf + journal_trip)
    #+ beta_Africa * Africa
    #+ beta_Europe * Europe 
    + beta_NorthAmerica_Oceania * (NorthAmerica + Oceania)
    #+ beta_Oceania * Oceania
   # + beta_NorthAmerica * NorthAmerica 
    + beta_Europe_SouthAmerica_Africa * (Europe + SouthAmerica + Africa)
    + beta_other_topics_DrivBeh * (Other + DrivBeh)
    + beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode * (AutoDriving + Covid19 + SocPoli + TrafficFlow + TransEmiss + TravMode)
    #+ beta_Covid19 * Covid19
    + beta_DataDriven * DataDriven
   # + beta_DrivBeh * DrivBeh
    #+ beta_ElectVeh * ElectVeh 
    #+ beta_PubTrans * PubTrans
    #+ beta_SocPoli * SocPoli
    #+ beta_TrafficFlow * TrafficFlow
    #+ beta_TransEmiss * TransEmiss
    #+ beta_TravMode * TravMode
    #+ beta_UrbEnv * UrbEnv
    )

v_no_code_av = 0

# CALCULATE THE LOGIT MODEL
V = {1: v_code_av, 0: v_no_code_av}
av = {1: 1, 0: 1}

logprob = models.loglogit(V, av, is_code_publicly_available)
biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'logit_code_av'

In [21]:
# ESTIMATE THE MODEL
results_code_av = biogeme.estimate()

print(results_code_av.printGeneralStatistics())


Number of estimated parameters:	9
Sample size:	10480
Excluded observations:	243
Init log likelihood:	-7264.182
Final log likelihood:	-1850.298
Likelihood ratio test for the init. model:	10827.77
Rho-square for the init. model:	0.745
Rho-square-bar for the init. model:	0.744
Akaike Information Criterion:	3718.596
Bayesian Information Criterion:	3783.912
Final gradient norm:	7.6094E-04
Nbr of threads:	10



In [22]:
results_code_av.getEstimatedParameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_ca,-3.892484,0.123694,-31.468582,0.000000e+00
beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode,-0.612644,0.113178,-5.413103,6.194186e-08
beta_DataDriven,0.738291,0.122535,6.025157,1.689457e-09
beta_Europe_SouthAmerica_Africa,1.372257,0.121731,11.272906,0.000000e+00
beta_NorthAmerica_Oceania,1.111752,0.125088,8.887750,0.000000e+00
beta_TRB_TRC,0.784953,0.103924,7.553145,4.241052e-14
beta_figure_number,0.019410,0.007375,2.631983,8.488823e-03
beta_other_topics_DrivBeh,-1.322422,0.247588,-5.341229,9.231874e-08
beta_paper_age_years,-0.177834,0.028439,-6.253092,4.024046e-10


In [23]:
#check the correlation between parameters
results_code_av.getCorrelationResults()

# Beta south America and North America are correlated 0.62 - they are extracted from the same variable

,Covariance,Correlation,t-test,p-value,Rob. cov.,Rob. corr.,Rob. t-test,Rob. p-value
beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode-asc_ca,-0.005667,-0.405779,16.516786,0.000000e+00,-0.006191,-0.442255,16.299227,0.000000e+00
beta_DataDriven-asc_ca,-0.004253,-0.277148,23.388020,0.000000e+00,-0.004457,-0.294044,23.380424,0.000000e+00
beta_DataDriven-beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode,0.005099,0.370603,10.229266,0.000000e+00,0.005072,0.365708,10.159829,0.000000e+00
beta_Europe_SouthAmerica_Africa-asc_ca,-0.009667,-0.636412,23.610459,0.000000e+00,-0.008850,-0.587746,24.075725,0.000000e+00
beta_Europe_SouthAmerica_Africa-beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode,-0.001448,-0.106338,11.416723,0.000000e+00,-0.001810,-0.131346,11.228882,0.000000e+00
beta_Europe_SouthAmerica_Africa-beta_DataDriven,-0.000119,-0.007986,3.649997,2.622436e-04,-0.000354,-0.023700,3.627694,2.859643e-04
beta_NorthAmerica_Oceania-asc_ca,-0.008895,-0.569002,22.594396,0.000000e+00,-0.008860,-0.572618,22.684002,0.000000e+00
beta_NorthAmerica_Oceania-beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode,-0.001341,-0.095684,9.811494,0.000000e+00,-0.000943,-0.066624,9.899420,0.000000e+00
beta_NorthAmerica_Oceania-beta_DataDriven,-0.000332,-0.021550,2.105253,3.526928e-02,-0.000497,-0.032395,2.099062,3.581148e-02
beta_NorthAmerica_Oceania-beta_Europe_SouthAmerica_Africa,0.009507,0.623735,-2.431036,1.505571e-02,0.009563,0.628039,-2.446400,1.442908e-02


In [24]:
# rho-bar squared
# results.data.numberOfFreeParameters
rhobar = 1 - (results_code_av.data.logLike - 8)/(results_const_code_av.data.logLike)
rhobar

0.1117976583482907

# 2. Choice model predicting the availability of data

In [25]:
# here we add the parameters for each alternative
asc_C_NR = Beta('asc_C_NR', 0, None, None, 0)
asc_NC_R = Beta('asc_NC_R', 0, None, None, 0)
asc_C_R = Beta('asc_C_R', 0, None, None, 0)

beta_number_of_authors = Beta('beta_number_of_authors', 0, None, None, 0)
beta_paper_age_years = Beta('beta_paper_age_years', 0, None, None, 0)
beta_open_access = Beta('beta_open_access', 0, None, None, 0)
beta_figure_number = Beta('beta_figure_number',  0, None, None, 0 )
beta_table_number = Beta('beta_table_number',  0, None, None, 0)
beta_reference_count = Beta('beta_reference_count',  0, None, None, 0)
beta_page_count = Beta('beta_page_count',  0, None, None, 0)

beta_number_of_authors_C_NR = Beta('beta_number_of_authors_C_NR', 0, None, None, 0)
beta_paper_age_years_C_NR = Beta('beta_paper_age_years_C_NR', 0, None, None, 0)
beta_open_access_C_NR = Beta('beta_open_access_C_NR', 0, None, None, 0)
beta_figure_number_C_NR = Beta('beta_figure_number_C_NR',  0, None, None, 0 )
beta_table_number_C_NR = Beta('beta_table_number_C_NR',  0, None, None, 0)
beta_reference_count_C_NR = Beta('beta_reference_count_C_NR',  0, None, None, 0)
beta_page_count_C_NR = Beta('beta_page_count_C_NR',  0, None, None, 0)
beta_review_time_days_C_NR = Beta('beta_review_time_days_C_NR',  0, None, None, 0) 
beta_review_time_days_missing_C_NR = Beta('beta_review_time_days_missing_C_NR',  0, None, None, 0)  

beta_number_of_authors_NC_R = Beta('beta_number_of_authors_NC_R', 0, None, None, 0)
beta_paper_age_years_NC_R = Beta('beta_paper_age_years_NC_R', 0, None, None, 0)
beta_open_access_NC_R = Beta('beta_open_access_NC_R', 0, None, None, 0)
beta_figure_number_NC_R = Beta('beta_figure_number_NC_R',  0, None, None, 0 )
beta_table_number_NC_R = Beta('beta_table_number_NC_R',  0, None, None, 0)
beta_reference_count_NC_R = Beta('beta_reference_count_NC_R',  0, None, None, 0)
beta_page_count_NC_R = Beta('beta_page_count_NC_R',  0, None, None, 0)
beta_review_time_days_NC_R = Beta('beta_review_time_days_NC_R',  0, None, None, 0) 
beta_review_time_days_missing_NC_R = Beta('beta_review_time_days_missing_NC_R',  0, None, None, 0)  

beta_number_of_authors_C_R = Beta('beta_number_of_authors_C_R', 0, None, None, 0)
beta_paper_age_years_C_R = Beta('beta_paper_age_years_C_R', 0, None, None, 0)
beta_open_access_C_R = Beta('beta_open_access_C_R', 0, None, None, 0)
beta_figure_number_C_R = Beta('beta_figure_number_C_R',  0, None, None, 0 )
beta_table_number_C_R = Beta('beta_table_number_C_R',  0, None, None, 0)
beta_reference_count_C_R = Beta('beta_reference_count_C_R',  0, None, None, 0)
beta_page_count_C_R = Beta('beta_page_count_C_R',  0, None, None, 0)
beta_review_time_days_C_R = Beta('beta_review_time_days_C_R',  0, None, None, 0) 
beta_review_time_days_missing_C_R = Beta('beta_review_time_days_missing_C_R',  0, None, None, 0)  

beta_paper_age_years_R =  Beta('beta_paper_age_years_R', 0, None, None, 0)

beta_table_number_NC_R_C_NR = Beta('beta_table_number_NC_R_C_NR', 0, None, None, 0)

beta_TRA = Beta('beta_TRA', 0, None, None, 0) # TRA too small to be used as basis - fix it
beta_TRB = Beta('beta_TRB', 0, None, None, 0)
beta_TRB_TRC = Beta('beta_TRB_TRC', 0, None, None, 0)
beta_TRB_TRC_TRE = Beta('beta_TRB_TRC_TRE', 0, None, None, 0)
beta_TRC = Beta('beta_TRC', 0, None, None, 0)
# beta_TRD = Beta('beta_TRD', 0, None, None, 0) reference category has the most common 
beta_TRF_TRIP = Beta('beta_TRF_TRIP', 0, None, None, 0)
beta_TRB_TRC_TRE_TRIP = Beta('beta_TRB_TRC_TRE_TRIP', 0, None, None, 0)
beta_TRE = Beta('beta_TRE', 0, None, None, 0)
beta_TRF = Beta('beta_TRF', 0, None, None, 0)
beta_TRIP = Beta('beta_TRIP', 0, None, None, 0)

beta_TRA_C_NR = Beta('beta_TRA_C_NR', 0, None, None, 0) # TRA too small to be used as basis - fix it
beta_TRB_C_NR  = Beta('beta_TRB_C_NR', 0, None, None, 0)
beta_TRC_C_NR  = Beta('beta_TRC_C_NR', 0, None, None, 0)
# beta_TRD = Beta('beta_TRD', 0, None, None, 0) reference category has the most common 
beta_TRE_C_NR  = Beta('beta_TRE_C_NR', 0, None, None, 0)
beta_TRF_C_NR  = Beta('beta_TRF_C_NR', 0, None, None, 0)
beta_TRIP_C_NR  = Beta('beta_TRIP_C_NR', 0, None, None, 0)
beta_TRB_TRE_C_NR = Beta('beta_TRB_TRE_C_NR', 0, None, None, 0)

beta_TRA_NC_R = Beta('beta_TRA_NC_R', 0, None, None, 0) # TRA too small to be used as basis - fix it
beta_TRB_NC_R  = Beta('beta_TRB_NC_R', 0, None, None, 0)
beta_TRC_NC_R  = Beta('beta_TRC_NC_R', 0, None, None, 0)
# beta_TRD = Beta('beta_TRD', 0, None, None, 0) reference category has the most common 
beta_TRE_NC_R  = Beta('beta_TRE_NC_R', 0, None, None, 0)
beta_TRF_NC_R  = Beta('beta_TRF_NC_R', 0, None, None, 0)
beta_TRIP_NC_R  = Beta('beta_TRIP_NC_R', 0, None, None, 0)

beta_TRA_C_R = Beta('beta_TRA_C_R', 0, None, None, 0) # TRA too small to be used as basis - fix it
beta_TRB_C_R  = Beta('beta_TRB_C_R', 0, None, None, 0)
beta_TRC_C_R  = Beta('beta_TRC_C_R', 0, None, None, 0)
# beta_TRD = Beta('beta_TRD', 0, None, None, 0) reference category has the most common 
beta_TRE_C_R  = Beta('beta_TRE_C_R', 0, None, None, 0)
beta_TRF_C_R  = Beta('beta_TRF_C_R', 0, None, None, 0)
beta_TRIP_C_R  = Beta('beta_TRIP_C_R', 0, None, None, 0)

beta_TRF_C  = Beta('beta_TRF_C', 0, None, None, 0)
beta_TRC_R  = Beta('beta_TRC_R', 0, None, None, 0)

# countries here # Asia is the baseline
beta_Africa_C_NR = Beta('beta_Africa_C_NR', 0, None, None, 0)
beta_Europe_C_NR = Beta('beta_Europe_C_NR', 0, None, None, 0)
beta_Europe_Oceania_C_NR = Beta('beta_Europe_Oceania_C_NR', 0, None, None, 0 )
beta_NorthAmerica_C_NR = Beta('beta_NorthAmerica_C_NR', 0, None, None, 0)
beta_Oceania_C_NR= Beta('beta_Oceania_C_NR', 0, None, None, 0)
beta_SouthAmerica_C_NR = Beta('beta_SouthAmerica_C_NR', 0, None, None, 0)
beta_Africa_NorthAmerica_C_NR = Beta('beta_Africa_NorthAmerica_C_NR', 0, None, None, 0)
beta_Africa_Europe_Oceania_C_NR = Beta('beta_Africa_Europe_Oceania_C_NR', 0, None, None, 0)

beta_Africa_NC_R = Beta('beta_Africa_NC_R', 0, None, None, 0)
beta_Europe_NC_R = Beta('beta_Europe_NC_R', 0, None, None, 0)
beta_NorthAmerica_NC_R = Beta('beta_NorthAmerica_NC_R', 0, None, None, 0)
beta_Oceania_NC_R= Beta('beta_Oceania_NC_R', 0, None, None, 0)
beta_SouthAmerica_NC_R = Beta('beta_SouthAmerica_NC_R', 0, None, None, 0)
beta_Europe_NorthAmerica_NC_R = Beta('beta_Europe_NorthAmerica_NC_R', 0, None, None, 0)

beta_Europe_C_R = Beta('beta_Europe_C_R', 0, None, None, 0)
beta_NorthAmerica_C_R = Beta('beta_NorthAmerica_C_R', 0, None, None, 0)
beta_Oceania_C_R= Beta('beta_Oceania_C_R', 0, None, None, 0)
beta_SouthAmerica_C_R = Beta('beta_SouthAmerica_C_R', 0, None, None, 0)
beta_Europe_NorthAmerica_C_R = Beta('beta_Europe_NorthAmerica_C_R', 0, None, None, 0)

beta_Europe_R = Beta('beta_Europe_R', 0, None, None, 0)
beta_NorthAmerica_R = Beta('beta_NorthAmerica_R', 0, None, None, 0)
beta_Oceania_R= Beta('beta_Oceania_R', 0, None, None, 0)
beta_SouthAmerica_R = Beta('beta_SouthAmerica_R', 0, None, None, 0)
beta_noAsia_R = Beta('beta_noAsia_R', 0, None, None, 0) # be careful Africa has no observations
beta_Oceania_SouthAmerica_R = Beta('beta_Oceania_SouthAmerica_R', 0, None, None, 0)
beta_Europe_NorthAmerica_R = Beta('beta_Europe_NorthAmerica_R', 0, None, None, 0)

# topics here - Optimisation is the baseline
beta_other_topics = Beta('beta_other_topics', 0, None, None, 0)
beta_AutoDriving = Beta('beta_AutoDriving', 0, None, None, 0)
beta_Covid19 = Beta('beta_Covid19', 0, None, None, 0)  
beta_DataDriven = Beta('beta_DataDriven', 0, None, None, 0)
beta_DrivBeh = Beta('beta_DrivBeh', 0, None, None, 0)
beta_ElectVeh = Beta('beta_ElectVeh', 0, None, None, 0)
beta_Optimization = Beta('beta_Optimization', 0, None, None, 0) 
beta_PubTrans = Beta('beta_PubTrans', 0, None, None, 0)
beta_SocPoli = Beta('beta_SocPoli', 0, None, None, 0)
beta_TrafficFlow = Beta('beta_TrafficFlow', 0, None, None, 0)
beta_TransEmiss = Beta('beta_TransEmiss', 0, None, None, 0)
beta_TravMode = Beta('beta_TravMode', 0, None, None, 0)
beta_UrbEnv = Beta('beta_UrbEnv', 0, None, None, 0)

beta_other_topics_DrivBeh = Beta('beta_other_topics_DrivBeh', 0, None, None, 0 )
beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode = Beta('beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode', 0, None, None, 0)

beta_other_topics_C_NR = Beta('beta_other_topics_C_NR', 0, None, None, 0)
beta_AutoDriving_C_NR = Beta('beta_AutoDriving_C_NR', 0, None, None, 0)
beta_Covid19_C_NR = Beta('beta_Covid19_C_NR', 0, None, None, 0)  
beta_DataDriven_C_NR = Beta('beta_DataDriven_C_NR', 0, None, None, 0)
beta_DrivBeh_C_NR = Beta('beta_DrivBeh_C_NR', 0, None, None, 0)
beta_ElectVeh_C_NR = Beta('beta_ElectVeh_C_NR', 0, None, None, 0)
beta_Optimization_C_NR = Beta('beta_Optimization_C_NR', 0, None, None, 0) 
beta_PubTrans_C_NR = Beta('beta_PubTrans_C_NR', 0, None, None, 0)
beta_SocPoli_C_NR = Beta('beta_SocPoli_C_NR', 0, None, None, 0)
beta_TrafficFlow_C_NR = Beta('beta_TrafficFlow_C_NR', 0, None, None, 0)
beta_TransEmiss_C_NR = Beta('beta_TransEmiss_C_NR', 0, None, None, 0)
beta_TravMode_C_NR = Beta('beta_TravMode_C_NR', 0, None, None, 0)
beta_UrbEnv_C_NR = Beta('beta_UrbEnv_C_NR', 0, None, None, 0)

beta_other_topics_NC_R = Beta('beta_other_topics_NC_R', 0, None, None, 0)
beta_AutoDriving_NC_R = Beta('beta_AutoDriving_NC_R', 0, None, None, 0)
beta_Covid19_NC_R = Beta('beta_Covid19_NC_R', 0, None, None, 0)  
beta_DataDriven_NC_R = Beta('beta_DataDriven_NC_R', 0, None, None, 0)
beta_DrivBeh_NC_R = Beta('beta_DrivBeh_NC_R', 0, None, None, 0)
beta_ElectVeh_NC_R = Beta('beta_ElectVeh_NC_R', 0, None, None, 0)
beta_Optimization_NC_R = Beta('beta_Optimization_NC_R', 0, None, None, 0) 
beta_PubTrans_NC_R = Beta('beta_PubTrans_NC_R', 0, None, None, 0)
beta_SocPoli_NC_R = Beta('beta_SocPoli_NC_R', 0, None, None, 0)
beta_TrafficFlow_NC_R = Beta('beta_TrafficFlow_NC_R', 0, None, None, 0)
beta_TransEmiss_NC_R = Beta('beta_TransEmiss_NC_R', 0, None, None, 0)
beta_TravMode_NC_R = Beta('beta_TravMode_NC_R', 0, None, None, 0)
beta_UrbEnv_NC_R = Beta('beta_UrbEnv_NC_R', 0, None, None, 0)

beta_other_topics_C_R = Beta('beta_other_topics_C_R', 0, None, None, 0)
beta_AutoDriving_C_R = Beta('beta_AutoDriving_C_R', 0, None, None, 0)
beta_Covid19_C_R = Beta('beta_Covid19_C_R', 0, None, None, 0)  
beta_DataDriven_C_R = Beta('beta_DataDriven_C_R', 0, None, None, 0)
beta_DrivBeh_C_R = Beta('beta_DrivBeh_C_R', 0, None, None, 0)
beta_ElectVeh_C_R = Beta('beta_ElectVeh_C_R', 0, None, None, 0)
beta_Optimization_C_R = Beta('beta_Optimization_C_R', 0, None, None, 0) 
beta_PubTrans_C_R = Beta('beta_PubTrans_C_R', 0, None, None, 0)
beta_SocPoli_C_R = Beta('beta_SocPoli_C_R', 0, None, None, 0)
beta_TrafficFlow_C_R = Beta('beta_TrafficFlow_C_R', 0, None, None, 0)
beta_TransEmiss_C_R = Beta('beta_TransEmiss_C_R', 0, None, None, 0)
beta_TravMode_C_R = Beta('beta_TravMode_C_R', 0, None, None, 0)
beta_UrbEnv_C_R = Beta('beta_UrbEnv_C_R', 0, None, None, 0)

beta_other_topics_R = Beta('beta_other_topics_R', 0, None, None, 0)
beta_AutoDriving_R = Beta('beta_AutoDriving_R', 0, None, None, 0)
beta_Covid19_R = Beta('beta_Covid19_R', 0, None, None, 0)  
beta_DataDriven_R = Beta('beta_DataDriven_R', 0, None, None, 0)
beta_DrivBeh_R = Beta('beta_DrivBeh_R', 0, None, None, 0)
beta_ElectVeh_R = Beta('beta_ElectVeh_R', 0, None, None, 0)
beta_Optimization_R = Beta('beta_Optimization_R', 0, None, None, 0) 
beta_PubTrans_R = Beta('beta_PubTrans_R', 0, None, None, 0)
beta_SocPoli_R = Beta('beta_SocPoli_R', 0, None, None, 0)
beta_TrafficFlow_R = Beta('beta_TrafficFlow_R', 0, None, None, 0)
beta_TransEmiss_R = Beta('beta_TransEmiss_R', 0, None, None, 0)
beta_TravMode_R = Beta('beta_TravMode_R', 0, None, None, 0)
beta_UrbEnv_R = Beta('beta_UrbEnv_R', 0, None, None, 0)

beta_PubTrans_C = Beta('beta_PubTrans_C', 0, None, None, 0)

beta_SocPoli_TraFlow = Beta('beta_SocPoli_TraFlow', 0, None, None, 0)
beta_Covid19_UrbEnv_NC_R = Beta('beta_Covid19_UrbEnv_NC_R', 0, None, None, 0)
beta_Covid19_UrbEnv_C_NR = Beta('beta_Covid19_UrbEnv_C_NR', 0, None, None, 0)
beta_Covid19_UrbEnv_C_R = Beta('beta_Covid19_UrbEnv_C_R', 0, None, None, 0)
beta_DrivBeh_TravMode_C_NR = Beta('beta_DrivBeh_TravMode_C_NR', 0, None, None, 0)

beta_TravMode_other_topics_R = Beta('beta_TravMode_other_topics_R', 0, None, None, 0)
beta_TravMode_other_topics_C_NR = Beta('beta_TravMode_other_topics_C_NR', 0, None, None, 0)


In [26]:
# add here any extra variable
data_available = Variable('data_available')

In [27]:
# Studies that are not quantitative are excluded
exclude = (is_quantitative_study != 1)
database.remove(exclude)

## 2.1 Model with only a constant

In [28]:
# see the previous specification

v_data_noCit_noRepo = 0

v_data_Cit_noRepo = (asc_C_NR)

v_data_noCit_Repo = (asc_NC_R)

v_data_Cit_Repo = (asc_C_R)

# CALCULATE THE LOGIT MODEL
V = {1: v_data_noCit_noRepo, 2: v_data_Cit_noRepo, 3: v_data_noCit_Repo, 4: v_data_Cit_Repo}
av = {1: 1, 2: 1, 3: 1, 4: 1}

logprob = models.loglogit(V, av, data_available)
biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'logit_data_repo_cit_const'

In [29]:
# ESTIMATE THE MODEL 
results_const_data_repo_cit = biogeme.estimate()

print(results_const_data_repo_cit.printGeneralStatistics())

Number of estimated parameters:	3
Sample size:	10480
Excluded observations:	0
Init log likelihood:	-14528.36
Final log likelihood:	-7895.009
Likelihood ratio test for the init. model:	13266.71
Rho-square for the init. model:	0.457
Rho-square-bar for the init. model:	0.456
Akaike Information Criterion:	15796.02
Bayesian Information Criterion:	15817.79
Final gradient norm:	3.4932E-03
Nbr of threads:	10



In [30]:
# check if need to exclude observations 
results_const_data_repo_cit.getEstimatedParameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_C_NR,-0.907581,0.021976,-41.297817,0.0
asc_C_R,-4.199951,0.096941,-43.324876,0.0
asc_NC_R,-3.306165,0.062664,-52.760500,0.0


## 2.2 Final specification

In [31]:
# see the previous specification

v_data_noCit_noRepo = 0

v_data_Cit_noRepo = (asc_C_NR # + beta_year * year_cent
   + beta_number_of_authors_C_NR * number_of_authors_cent
    + beta_paper_age_years_C_NR * paper_age_years_cent
    + beta_figure_number * figure_number_cent
    + beta_table_number_NC_R_C_NR * table_number_cent
   # + beta_reference_count_C_NR * reference_count_cent
    + beta_page_count * page_count_cent
   # + beta_TRA_C_NR * journal_tra
    + beta_TRB_TRE_C_NR * (journal_trb + journal_tre)
   # + beta_TRC_C_NR * journal_trc 
    #+ beta_TRD * journal_trd
    #+ beta_TRE_C_NR * journal_tre # 
    + beta_TRF_C * journal_trf 
    #+ beta_TRIP_C_NR * journal_trip 
    + beta_Africa_NorthAmerica_C_NR * (Africa + NorthAmerica) 
    #+ beta_Asia * Asia # baseline as the most frequent
    + beta_Europe_Oceania_C_NR * (Europe + Oceania) 
    #+ beta_Africa_Europe_Oceania_C_NR * (Africa + Europe + Oceania) 
    #+ beta_NorthAmerica_Oceania * (NorthAmerica + Oceania)
   # + beta_Oceania_C_NR * Oceania
    #+ beta_NorthAmerica_C_NR * NorthAmerica 
   # + beta_SouthAmerica_C_NR * SouthAmerica 
    #+ beta_Europe_SouthAmerica_Africa * (Europe + SouthAmerica + Africa)
    + beta_TravMode_other_topics_C_NR * (TravMode + Other)
    + beta_AutoDriving_C_NR * AutoDriving
    + beta_Covid19_UrbEnv_C_NR * (Covid19 + UrbEnv)
    + beta_DataDriven_C_NR * DataDriven
    + beta_DrivBeh * (DrivBeh)
    #+ beta_ElectVeh_C_NR * ElectVeh
    # + beta_Optimization * Optimization # this is the baseline
    #+ beta_PubTrans_L * PubTrans
    + beta_SocPoli_TraFlow * (SocPoli + TrafficFlow) 
   # + beta_TrafficFlow * TrafficFlow
    + beta_TransEmiss_C_NR * TransEmiss
   # + beta_TravMode_C_NR * TravMode
    #+ beta_TravMode * TravMode 
    #+ beta_UrbEnv_C_NR * UrbEnv
    )

v_data_noCit_Repo = (asc_NC_R # + beta_year * year_cent
    #+ beta_number_of_authors_NC_R * number_of_authors_cent
    + beta_paper_age_years_R * paper_age_years_cent
    + beta_figure_number * figure_number_cent
    + beta_table_number_NC_R_C_NR * table_number_cent
    #+ beta_reference_count_NC_R * reference_count_cent
    + beta_page_count * page_count_cent
    #+ beta_TRA_NC_R * journal_tra
    #+ beta_TRB_TRE_NC_R * journal_trb  
    + beta_TRC_R * journal_trc 
    #+ beta_TRD * journal_trd
    #+ beta_TRE_NC_R * journal_tre 
    #+ beta_TRF_NC_R * journal_trf 
    #+ beta_TRIP_NC_R * journal_trip 
    #+ beta_Africa * Africa 
    #+ beta_Asia * Asia # baseline as the most frequent
    + beta_Europe_NorthAmerica_R * (Europe + NorthAmerica) 
    #+ beta_NorthAmerica_Oceania * (NorthAmerica + Oceania)
    + beta_Oceania_SouthAmerica_R * (Oceania + SouthAmerica)
    #+ beta_noAsia_R * (1 - Asia)
    #+ beta_NorthAmerica_R * NorthAmerica 
    #+ beta_SouthAmerica_R * SouthAmerica 
    #+ beta_Europe_SouthAmerica_Africa * (Europe + SouthAmerica + Africa)
    + beta_TravMode_other_topics_R * (TravMode + Other)
   # + beta_AutoDriving_NC_R * AutoDriving
   # + beta_Covid19_UrbEnv_NC_R * (Covid19 + UrbEnv)
    + beta_DataDriven_NC_R * DataDriven
    #+ beta_DataDriven_R * DataDriven
    + beta_DrivBeh * (DrivBeh)
    #+ beta_ElectVeh_R * ElectVeh
    # + beta_Optimization * Optimization # this is the baseline
    # + beta_PubTrans_NC_R * PubTrans
     + beta_SocPoli_TraFlow * (SocPoli + TrafficFlow) 
   # + beta_TrafficFlow * TrafficFlow 
     + beta_TransEmiss_R * TransEmiss
    # + beta_TravMode_R * TravMode
    # + beta_TravMode * TravMode 
    # + beta_UrbEnv_NC_R * UrbEnv
    )

v_data_Cit_Repo = (asc_C_R # + beta_year * year_cent
   #+ beta_number_of_authors_C_R * number_of_authors_cent
    + beta_paper_age_years_R * paper_age_years_cent
    + beta_figure_number * figure_number_cent
    #+ beta_table_number_C_R * table_number_cent
   # + beta_reference_count_C_R * reference_count_cent
    + beta_page_count * page_count_cent
    #+ beta_TRA_C_R * journal_tra
    #+ beta_TRB_C_R * journal_trb 
    + beta_TRC_R * journal_trc 
    #+ beta_TRD * journal_trd
   # + beta_TRE_C_R * journal_tre
    + beta_TRF_C * journal_trf 
    + beta_TRIP_C_R * journal_trip 
    #+ beta_Africa * Africa  # no observations here 
    #+ beta_Asia * Asia # baseline as the most frequent
    + beta_Europe_NorthAmerica_R * (Europe + NorthAmerica) 
    #+ beta_NorthAmerica_Oceania * (NorthAmerica + Oceania)
    + beta_Oceania_SouthAmerica_R * (Oceania + SouthAmerica)
    #+ beta_noAsia_R * (1 - Asia)
    #+ beta_NorthAmerica_R * NorthAmerica
    #+ beta_SouthAmerica_R * SouthAmerica
    #+ beta_Europe_SouthAmerica_Africa * (Europe + SouthAmerica + Africa)
    + beta_TravMode_other_topics_R * (TravMode + Other) 
    #+ beta_AutoDriving_C_R * AutoDriving
    + beta_Covid19_UrbEnv_C_R * (Covid19 + UrbEnv)
    + beta_DataDriven_C_R * DataDriven
   # + beta_DataDriven_R * DataDriven
    + beta_DrivBeh * DrivBeh
    #+ beta_ElectVeh_R * ElectVeh
    # + beta_Optimization * Optimization # this is the baseline
    #+ beta_PubTrans_L * PubTrans
    + beta_SocPoli_TraFlow * (SocPoli + TrafficFlow) 
   # + beta_TrafficFlow * TrafficFlow
    + beta_TransEmiss_R * TransEmiss 
    # + beta_TravMode_R * TravMode
   # + beta_TravMode * TravMode 
    #+ beta_UrbEnv_C_R * UrbEnv
    )

# CALCULATE THE LOGIT MODEL
V = {1: v_data_noCit_noRepo, 2: v_data_Cit_noRepo, 3: v_data_noCit_Repo, 4: v_data_Cit_Repo}
av = {1: 1, 2: 1, 3: 1, 4: 1}

logprob = models.loglogit(V, av, data_available)
biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'logit_data_repo_cit'

In [32]:
# ESTIMATE THE MODEL 
results_data_repo_cit = biogeme.estimate()

print(results_data_repo_cit.printGeneralStatistics())

Number of estimated parameters:	29
Sample size:	10480
Excluded observations:	0
Init log likelihood:	-14528.36
Final log likelihood:	-7145.929
Likelihood ratio test for the init. model:	14764.87
Rho-square for the init. model:	0.508
Rho-square-bar for the init. model:	0.506
Akaike Information Criterion:	14349.86
Bayesian Information Criterion:	14560.32
Final gradient norm:	1.1399E-02
Nbr of threads:	10



In [33]:
# check if need to exclude observations 
results_data_repo_cit.getEstimatedParameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_C_NR,-0.892217,0.057931,-15.401265,0.000000e+00
asc_C_R,-5.600350,0.192746,-29.055638,0.000000e+00
asc_NC_R,-4.460605,0.149171,-29.902663,0.000000e+00
beta_Africa_NorthAmerica_C_NR,0.769492,0.059188,13.000793,0.000000e+00
beta_AutoDriving_C_NR,-1.045369,0.139628,-7.486813,7.061018e-14
beta_Covid19_UrbEnv_C_NR,0.443479,0.081196,5.461856,4.711822e-08
beta_Covid19_UrbEnv_C_R,1.164834,0.283763,4.104957,4.043900e-05
beta_DataDriven_C_NR,0.355307,0.086655,4.100237,4.127274e-05
beta_DataDriven_C_R,1.829374,0.226976,8.059780,6.661338e-16
beta_DataDriven_NC_R,0.879045,0.173586,5.064031,4.104838e-07


In [34]:
#check the correlation between parameters
results_data_repo_cit.getCorrelationResults()

,Covariance,Correlation,t-test,p-value,Rob. cov.,Rob. corr.,Rob. t-test,Rob. p-value
asc_C_R-asc_C_NR,1.067562e-03,0.090596,-22.914532,0.000000e+00,0.001062,0.095127,-24.031546,0.000000e+00
asc_NC_R-asc_C_NR,9.225202e-04,0.107610,-23.415181,0.000000e+00,0.000893,0.103366,-23.120001,0.000000e+00
asc_NC_R-asc_C_R,1.714254e-02,0.575259,6.768430,1.301870e-11,0.015220,0.529350,6.697078,2.126277e-11
beta_Africa_NorthAmerica_C_NR-asc_C_NR,-1.472270e-03,-0.427028,16.750832,0.000000e+00,-0.001449,-0.422529,16.822872,0.000000e+00
beta_Africa_NorthAmerica_C_NR-asc_C_R,-3.920096e-04,-0.032710,29.944175,0.000000e+00,-0.000498,-0.043630,31.212106,0.000000e+00
...,...,...,...,...,...,...,...,...
beta_table_number_NC_R_C_NR-beta_figure_number,8.314935e-06,0.284357,-0.626659,5.308828e-01,0.000009,0.286772,-0.620390,5.350010e-01
beta_table_number_NC_R_C_NR-beta_number_of_authors_C_NR,-6.028125e-07,-0.007309,3.492737,4.780977e-04,-0.000001,-0.015467,3.485612,4.910131e-04
beta_table_number_NC_R_C_NR-beta_page_count,-1.608461e-05,-0.484869,5.851549,4.870153e-09,-0.000016,-0.468906,5.776206,7.640385e-09
beta_table_number_NC_R_C_NR-beta_paper_age_years_C_NR,-2.806386e-06,-0.034330,-3.523086,4.265523e-04,-0.000002,-0.025574,-3.555821,3.768007e-04


In [35]:
# rho-bar squared
# results.data.numberOfFreeParameters
rhobar = 1 - (results_data_repo_cit.data.logLike - 26)/(results_const_data_repo_cit.data.logLike)
rhobar

0.0915869768134514

# 3. Compile latex tables with the estimation results

In [36]:
# I need to add the significance levels with the *** also
# I need to add the stats of the models in latex format 

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

In [37]:
# here is the function to build a table with one column for the estimates and standard errors or t-tests

def build_table_single_col(df, variable_order, labels=None, stat_type="stderr"):
    labels = labels or {}
    val_col = "Value"
    stat_col = "Rob. Std err" if stat_type == "stderr" else "Rob. t-test"
    pval_col = "Rob. p-value"

    lines = [r"\begin{tabular}{lc}",
             r"\toprule",
             r"Variable & Estimate \\",
             r"\midrule"]

    for var in variable_order:
        est = df.loc[var, val_col]
        stat = df.loc[var, stat_col]
        p = df.loc[var, pval_col]
        s = stars(p)
        cell = f"{est:.3g} ({stat:.3g})"
        if s:
            cell += f" {s}"
        lines.append(f"{labels.get(var, var)} & {cell}" + r" \\")

    lines += [r"\bottomrule", r"\end{tabular}"]
    lines.append(r"\vspace{2pt}")
    lines.append(r"\\ {\footnotesize *** p $<$ 0.001, ** p $<$ 0.01, * p $<$ 0.05}")
    return "\n".join(lines)

In [38]:
# here is the function to build the table for the latex output

def build_table(df, column_order, variable_order, param_lookup,
                 labels=None, stat_type="stderr"):
    labels = labels or {}
    val_col = "Value"
    stat_col = "Rob. Std err" if stat_type == "stderr" else "Rob. t-test"
    pval_col = "Rob. p-value"

    lines = [r"\begin{tabular}{l" + "c" * len(column_order) + "}",
             r"\toprule",
             "Variable & " + " & ".join(column_order) + r" \\",
             r"\midrule"]

    for base in variable_order:
        col_params = param_lookup.get(base, {})
        row = [labels.get(base, base)]
        for c in column_order:
            param = col_params.get(c)
            if param is None:
                row.append("--")
            else:
                est = df.loc[param, val_col]
                stat = df.loc[param, stat_col]
                p = df.loc[param, pval_col]
                s = stars(p)
                cell = f"{est:.3g} ({stat:.3g})"
                if s:
                    cell += f" {s}"
                row.append(cell)
        lines.append(" & ".join(row) + r" \\")

    lines += [r"\bottomrule", r"\end{tabular}"]
    lines.append(r"\vspace{2pt}")
    lines.append(r"\\ {\footnotesize *** p $<$ 0.001, ** p $<$ 0.01, * p $<$ 0.05}")
    return "\n".join(lines)

In [39]:
# here is the function to classify the variable names and extract the base name
def classify_and_base(var):
    tokens = var.strip().split("_")
    matched = set()
    used = [False] * len(tokens)

    i = 0
    while i < len(tokens) - 1:
        pair = (tokens[i], tokens[i+1])
        if pair == ("C", "NR"):
            matched.add("C_NR"); used[i] = used[i+1] = True; i += 2; continue
        if pair == ("NC", "R"):
            matched.add("NC_R"); used[i] = used[i+1] = True; i += 2; continue
        if pair == ("C", "R"):
            matched.add("C_R"); used[i] = used[i+1] = True; i += 2; continue
        i += 1

    if not matched:
        for j, tok in enumerate(tokens):
            if tok == "C" and not used[j]:
                matched.update(["C_NR", "C_R"]); used[j] = True; break
        if not matched:
            for j, tok in enumerate(tokens):
                if tok == "R" and not used[j]:
                    matched.update(["C_R", "NC_R"]); used[j] = True; break

    base_tokens = [t for t, u in zip(tokens, used) if not u]
    base_name = "_".join(base_tokens) if base_tokens else var

    if not matched:
        matched = {"C_NR", "C_R", "NC_R"}

    return base_name, sorted(matched)

In [40]:
# save the model statistics in a LaTeX table format

def build_stats_table_from_results(results_full, results_const_only, caption, label):
    stats_full = results_full.getGeneralStatistics()
    stats_const = results_const_only.getGeneralStatistics()

    K_full = stats_full["Number of estimated parameters"][0]
    K_const = stats_const["Number of estimated parameters"][0]
    K = K_full - K_const

    N = stats_full["Sample size"][0]

    LLc = stats_const["Final log likelihood"][0]   # constants-only model's converged LL
    LLbeta = stats_full["Final log likelihood"][0]  # full model's converged LL

    rho2 = 1 - (LLbeta - K) / LLc

    def fmt_int(x):
        return f"{x:,.0f}"

    def fmt_num(x, decimals=3):
        return f"{x:,.{decimals}g}"

    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        r"\small",
        r"\begin{tabular}{lr}",
        r"\toprule",
        r"\textbf{Statistic} & \textbf{Value} \\",
        r"\midrule",
        rf"Number of parameters associated with explanatory variables ($K$) & {K} \\",
        rf"Number of observations & {fmt_int(N)} \\",
        rf"Constant log likelihood ($L(c)$) & {fmt_int(LLc)} \\",
        rf"Final log likelihood  ($L(\hat{{\beta}}) $) & {fmt_int(LLbeta)} \\",
        r"Adjusted likelihood ratio index $\rho^2 = 1 - \frac{L(\hat{\beta}) - K}{L(c)}$ & "
        + fmt_num(rho2) + r" \\",
        r"\bottomrule",
        r"\end{tabular}",
        rf"\label{{{label}}}",
        r"\end{table}",
    ]
    return "\n".join(lines)

## 3.1 Generate the tables for code availability

In [41]:
df_code = results_code_av.getEstimatedParameters()

In [42]:
df_code.index.tolist()

['asc_ca',
 'beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode',
 'beta_DataDriven',
 'beta_Europe_SouthAmerica_Africa',
 'beta_NorthAmerica_Oceania',
 'beta_TRB_TRC',
 'beta_figure_number',
 'beta_other_topics_DrivBeh',
 'beta_paper_age_years']

In [43]:
tex = build_table_single_col(
    df=df_code,
    variable_order=["asc_ca",
                    "beta_TRB_TRC",
                    "beta_Europe_SouthAmerica_Africa", "beta_NorthAmerica_Oceania",
                    "beta_DataDriven", "beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode",
                    "beta_other_topics_DrivBeh", 
                    "beta_figure_number", "beta_paper_age_years"
                    ]
)
print(tex)

\begin{tabular}{lc}
\toprule
Variable & Estimate \\
\midrule
asc_ca & -3.89 (0.124) *** \\
beta_TRB_TRC & 0.785 (0.104) *** \\
beta_Europe_SouthAmerica_Africa & 1.37 (0.122) *** \\
beta_NorthAmerica_Oceania & 1.11 (0.125) *** \\
beta_DataDriven & 0.738 (0.123) *** \\
beta_AutoDriving_Covid19_SocPoli_TrafficFlow_TransEmiss_TravMode & -0.613 (0.113) *** \\
beta_other_topics_DrivBeh & -1.32 (0.248) *** \\
beta_figure_number & 0.0194 (0.00737) ** \\
beta_paper_age_years & -0.178 (0.0284) *** \\
\bottomrule
\end{tabular}
\vspace{2pt}
\\ {\footnotesize *** p $<$ 0.001, ** p $<$ 0.01, * p $<$ 0.05}


In [44]:
# model statistics
tex_stats_code = build_stats_table_from_results(
    results_full=results_code_av,
    results_const_only=results_const_code_av,
    caption="Statistics for the choice model predicting the availability of code",
    label="tab:model_statistics_code",
)
print(tex_stats_code)

\begin{table}[htbp]
\centering
\caption{Statistics for the choice model predicting the availability of code}
\small
\begin{tabular}{lr}
\toprule
\textbf{Statistic} & \textbf{Value} \\
\midrule
Number of parameters associated with explanatory variables ($K$) & 8 \\
Number of observations & 10,480 \\
Constant log likelihood ($L(c)$) & -2,092 \\
Final log likelihood  ($L(\hat{\beta}) $) & -1,850 \\
Adjusted likelihood ratio index $\rho^2 = 1 - \frac{L(\hat{\beta}) - K}{L(c)}$ & 0.112 \\
\bottomrule
\end{tabular}
\label{tab:model_statistics_code}
\end{table}


## 3.2 Generate the tables for data availability

In [45]:
df_data = results_data_repo_cit.getEstimatedParameters()

In [46]:
df_data.index = df_data.index.str.strip()

param_lookup = {}   # base_name -> {column: actual_param_name}
for var in df_data.index.tolist():
    base, cols = classify_and_base(var)
    param_lookup.setdefault(base, {})
    for c in cols:
        param_lookup[base][c] = var

In [47]:
for base, cols in param_lookup.items():
    print(f"{base:20s} -> {cols}")

asc                  -> {'C_NR': 'asc_C_NR', 'C_R': 'asc_C_R', 'NC_R': 'asc_NC_R'}
beta_Africa_NorthAmerica -> {'C_NR': 'beta_Africa_NorthAmerica_C_NR'}
beta_AutoDriving     -> {'C_NR': 'beta_AutoDriving_C_NR'}
beta_Covid19_UrbEnv  -> {'C_NR': 'beta_Covid19_UrbEnv_C_NR', 'C_R': 'beta_Covid19_UrbEnv_C_R'}
beta_DataDriven      -> {'C_NR': 'beta_DataDriven_C_NR', 'C_R': 'beta_DataDriven_C_R', 'NC_R': 'beta_DataDriven_NC_R'}
beta_DrivBeh         -> {'C_NR': 'beta_DrivBeh', 'C_R': 'beta_DrivBeh', 'NC_R': 'beta_DrivBeh'}
beta_Europe_NorthAmerica -> {'C_R': 'beta_Europe_NorthAmerica_R', 'NC_R': 'beta_Europe_NorthAmerica_R'}
beta_Europe_Oceania  -> {'C_NR': 'beta_Europe_Oceania_C_NR'}
beta_Oceania_SouthAmerica -> {'C_R': 'beta_Oceania_SouthAmerica_R', 'NC_R': 'beta_Oceania_SouthAmerica_R'}
beta_SocPoli_TraFlow -> {'C_NR': 'beta_SocPoli_TraFlow', 'C_R': 'beta_SocPoli_TraFlow', 'NC_R': 'beta_SocPoli_TraFlow'}
beta_TRB_TRE         -> {'C_NR': 'beta_TRB_TRE_C_NR'}
beta_TRC             -> {'C_R': '

In [48]:
tex = build_table(
    df=df_data,
    column_order=["C_NR", "NC_R", "C_R"],
    variable_order=["asc",
                    "beta_TRB_TRE", "beta_TRC", "beta_TRF", "beta_TRIP", 
                    "beta_Africa_NorthAmerica", "beta_Europe_Oceania", "beta_Europe_NorthAmerica", "beta_Oceania_SouthAmerica",
                    "beta_DataDriven", "beta_Covid19_UrbEnv",
                    "beta_TransEmiss", "beta_DrivBeh", "beta_SocPoli_TraFlow","beta_AutoDriving", "beta_TravMode_other_topics",
                    "beta_figure_number", "beta_table_number", "beta_page_count", "beta_number_of_authors", "beta_paper_age_years"
                    ],
    param_lookup=param_lookup
)
print(tex)

\begin{tabular}{lccc}
\toprule
Variable & C_NR & NC_R & C_R \\
\midrule
asc & -0.892 (0.0579) *** & -4.46 (0.149) *** & -5.6 (0.193) *** \\
beta_TRB_TRE & -0.263 (0.064) *** & -- & -- \\
beta_TRC & -- & 0.556 (0.128) *** & 0.556 (0.128) *** \\
beta_TRF & -1.23 (0.117) *** & -- & -1.23 (0.117) *** \\
beta_TRIP & -- & -- & -0.86 (0.431) * \\
beta_Africa_NorthAmerica & 0.769 (0.0592) *** & -- & -- \\
beta_Europe_Oceania & 0.351 (0.056) *** & -- & -- \\
beta_Europe_NorthAmerica & -- & 1.68 (0.142) *** & 1.68 (0.142) *** \\
beta_Oceania_SouthAmerica & -- & 1.17 (0.237) *** & 1.17 (0.237) *** \\
beta_DataDriven & 0.355 (0.0867) *** & 0.879 (0.174) *** & 1.83 (0.227) *** \\
beta_Covid19_UrbEnv & 0.443 (0.0812) *** & -- & 1.16 (0.284) *** \\
beta_TransEmiss & 0.335 (0.0862) *** & -0.525 (0.275) & -0.525 (0.275) \\
beta_DrivBeh & -0.297 (0.125) * & -0.297 (0.125) * & -0.297 (0.125) * \\
beta_SocPoli_TraFlow & -0.628 (0.0751) *** & -0.628 (0.0751) *** & -0.628 (0.0751) *** \\
beta_AutoDriving & 

In [49]:
tex_stats_data = build_stats_table_from_results(
    results_full=results_data_repo_cit,
    results_const_only=results_const_data_repo_cit,
    caption="Statistics for the choice model predicting the availability of data",
    label="tab:model_statistics_data",
)
print(tex_stats_data)

\begin{table}[htbp]
\centering
\caption{Statistics for the choice model predicting the availability of data}
\small
\begin{tabular}{lr}
\toprule
\textbf{Statistic} & \textbf{Value} \\
\midrule
Number of parameters associated with explanatory variables ($K$) & 26 \\
Number of observations & 10,480 \\
Constant log likelihood ($L(c)$) & -7,895 \\
Final log likelihood  ($L(\hat{\beta}) $) & -7,146 \\
Adjusted likelihood ratio index $\rho^2 = 1 - \frac{L(\hat{\beta}) - K}{L(c)}$ & 0.0916 \\
\bottomrule
\end{tabular}
\label{tab:model_statistics_data}
\end{table}
